In [2]:

%pip install \
    "strands-agents>=1.60.0,<2.0.0" \
    "strands-agents-tools>=0.8.9,<0.9.0" \
    "langchain>=1.4.0,<2.0.0" \
    "langchain-aws>=1.8.0,<2.0.0" \
    "langgraph>=1.2.0,<2.0.0" \
    "boto3>=1.40.0"


INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.5/923.5 kB 9.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 33.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 21.9 MB/s  0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-con

In [8]:

from strands import Agent
from strands.models.openai import OpenAIModel
import os
from google.colab import userdata

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": userdata.get("GROQ_API_KEY"),
        "base_url": os.getenv(
            "GROQ_BASE_URL",
            "https://api.groq.com/openai/v1"
        ),
        "max_retries": 5,
    },
    model_id=MODEL_ID,
    params={"extra_body": {"include_reasoning": False}},
)

agent = Agent(model=model, callback_handler=None )

response = agent(
    """
      Explain groq in a bulleted list.
    """
)

print(response)


**GROQ (Graph‑Relational Object Queries) – Quick Overview**

- **What it is**
  - A query language created by **Sanity.io** for fetching and transforming content stored in a Sanity dataset.
  - Designed to be **human‑readable**, **declarative**, and **flexible** for both simple and complex queries.

- **Core concepts**
  - **Documents**: the primary data units (e.g., `post`, `author`, `product`).
  - **Fields**: key‑value pairs inside a document, which can be scalars, objects, arrays, or references.
  - **References**: links between documents, resolved on‑the‑fly via the `->` operator.
  - **Projection**: selecting which fields to return and shaping the output (similar to `SELECT` in SQL).

- **Basic syntax**
  - `*` – selects **all documents** (or all of a certain type when filtered).
  - `[condition]` – **filter** clause (e.g., `[_type == "post"]`).
  - `{ field1, field2, "alias": field3 }` – **projection** (what to return).
  - `->` – dereferences a reference (e.g., `author->name`).

In Strands Agents, callback_handler=None tells the agent not to use a callback handler for processing events during execution. In your case, it prevents the default handler from printing streamed response text automatically, so print(response) doesn't display the same answer a second time


In [23]:
from datetime import datetime
from zoneinfo import ZoneInfo
from strands_tools import calculator

from strands import tool


@tool
def current_time(timezone: str = "Asia/Kolkata") -> str:
    """Get the current date and time in a given IANA timezone.

    Args:
        timezone: IANA timezone name, for example "Asia/Kolkata" or "America/New_York".
    """
    return datetime.now(ZoneInfo(timezone)).strftime("%A, %d %B %Y %I:%M %p %Z")


agent = Agent(model=model, tools=[calculator, current_time], callback_handler= None)

result = agent(
    "What time is it right now in Bangalore?"
    )

print(result.message["content"][0]["text"])



In Bangalore (which follows Indian Standard Time, IST), it is currently **11:46 PM on Thursday, 29 October 2026**.
